### Model Deployment
After we chose the best models and hyperparameters, now it is time to run the model for the last time and deploy it. 

Since this is a capstone project for research purposes, we will only store the model in our local directory. In real life, the model needs to be deployed into the server (cloud, on premises). 

For retrieval, we will use BeerRecommendationModelV2 class with hyperparameters epochs = 20 and learning_rate = 0.01

For ranking, we will use BeerRankingModelV1  with learning rate 0.01 with 30 epoch

For content based filtering, we will use cosine similarity matrix by using beer name, beer style and abv rate.

In [17]:
#Install tensor libraries
!pip install -q tensorflow
!pip install -q tensorflow-recommenders
!pip install -q tensorflow-ranking
!pip install -q numpy

import pandas as pd
#load dataframes from csv files created in previous steps
file_path = "dataset/ratebeer_dataset.csv"
df = pd.read_csv(file_path)
file_path_beers = "dataset/ratebeer_beers.csv"
df_beers = pd.read_csv(file_path_beers)

df.info()

import tensorflow as tf
import tensorflow_recommenders as tfrs
from typing import Dict, Text
# Tensorflow dataset for ratings
def get_ratings_in_tensor_dataset(df):
    #Drop review_text as it is quite large to create a tensor dataset
    df_without_review = df.drop(columns=['review_text'])
    ratings = tf.data.Dataset.from_tensor_slices(dict(df_without_review)).map(lambda x: {
        "beer_id": x["beer_id"],
        "beer_name": x["beer_name"],
        "brewer_id": x["brewer_id"],
        "beer_style": x["beer_style"],
        "beer_abv": x["beer_abv"],
        "review_username": x["review_username"],
        "review_overall": x["review_overall"],
        "review_taste": x["review_taste"],
        "review_palate": x["review_palate"],
        "review_appearance": x["review_appearance"],
        "review_aroma": x["review_aroma"]
    })
    return ratings

# Tensorflow dataset for beers
def get_beers_in_tensor_dataset(df_beers):
    beers = tf.data.Dataset.from_tensor_slices(dict(df_beers)).map(lambda x: {
        "beer_id": x["beer_id"],
        "beer_name": x["beer_name"],
        "beer_abv": x["beer_abv"],
        "beer_style": x["beer_style"],
        "brewer_id": x["brewer_id"]
    })
    return beers


# Converting in tensor dataset
ratings = get_ratings_in_tensor_dataset(df)
beers = get_beers_in_tensor_dataset(df_beers)

import numpy as np

def prepare_dictionary_for_embeddings(ratings, beers, embedding_dimension):
    # We need a vocabulary that maps a raw feature value to an integer in a contiguous range
    # this allows us to look up the corresponding embeddings in our embedding tables.
    usernames = ratings.batch(1_000).map(lambda x: x["review_username"])
    unique_usernames = np.unique(np.concatenate(list(usernames)))

    beer_ids = beers.batch(1_000).map(lambda x: x["beer_id"])
    unique_beer_ids = np.unique(np.concatenate(list(beer_ids)))

    # Now add also beer style into the retrieval model
    beer_styles = beers.batch(1_000).map(lambda x: x["beer_style"])
    unique_beer_styles = np.unique(np.concatenate(list(beer_styles)))

    # Now add also beer name into the retrieval model
    beer_names = beers.batch(1_000).map(lambda x: x["beer_name"])
    unique_beer_names = np.unique(np.concatenate(list(beer_styles)))

    #Add beer_abvs, this value will be used in the model too
    beer_abvs = np.concatenate(list(ratings.map(lambda x: x["beer_abv"]).batch(100)))

    max_beer_abv = beer_abvs.max()
    min_beer_abv = beer_abvs.min()

    beer_abv_buckets = np.linspace(
        min_beer_abv, max_beer_abv, num=1000,
    )
    return {
        "embedding_dimension": embedding_dimension,
        "unique_usernames": unique_usernames,
        "unique_beer_ids": unique_beer_ids,
        "unique_beer_styles": unique_beer_styles,
        "unique_beer_names": unique_beer_names,
        "beer_abv_buckets": beer_abv_buckets
    }

#Let's start with embedding size 32
embedding_dicts = prepare_dictionary_for_embeddings(ratings, beers, 32)

# We get all the data
sample_size = 308103

def get_train_test_cached(ratings, sample_size, batch_size=500):
    tf.random.set_seed(42)
    shuffled = ratings.shuffle(sample_size, seed=42, reshuffle_each_iteration=False)
    train_size = int(sample_size * 0.8)
    test_size = int(sample_size * 0.2)
    train = shuffled.take(train_size)
    test = shuffled.skip(train_size).take(test_size)
    print(f"The sample has a size of {sample_size}. Training size is {train_size}, test size is {test_size} and batch size is{batch_size}" )

    #Cache the dataset for better performance 
    cached_train = train.shuffle(train_size).batch(batch_size).cache()
    cached_test = test.batch(int(batch_size/4)).cache()
    return cached_train, cached_test

cached_train, cached_test = get_train_test_cached(ratings, sample_size, batch_size = 30000)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 308103 entries, 0 to 308102
Data columns (total 14 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   beer_name          308103 non-null  object 
 1   beer_id            308103 non-null  int64  
 2   brewer_id          308103 non-null  int64  
 3   beer_abv           308103 non-null  float64
 4   beer_style         308103 non-null  object 
 5   review_appearance  308103 non-null  float64
 6   review_aroma       308103 non-null  float64
 7   review_palate      308103 non-null  float64
 8   review_taste       308103 non-null  float64
 9   review_overall     308103 non-null  float64
 10  review_time        308103 non-null  int64  
 11  review_username    308103 non-null  object 
 12  review_text        307302 non-null  object 
 13  review_count       308103 non-null  int64  
dtypes: float64(6), int64(4), object(4)
memory usage: 32.9+ MB
The sample has a size of 308103. Training 

In [18]:
# Retrieval model version 2
# In this model, we also added beer style on the candidate model (beer)
# Query model : username / Candidate model: beer id and beer style
class UserRetrievalModelV2(tfrs.models.Model):
    def __init__(self,unique_usernames, embedding_dimension):
        super().__init__()
        self.user_model: tf.keras.Model = tf.keras.Sequential([
            tf.keras.layers.StringLookup(vocabulary=unique_usernames, mask_token=None, name="username_lookup"),
            tf.keras.layers.Embedding(len(unique_usernames) + 1, embedding_dimension, name="username_embedding")
        ], name="user_model_sequential")

    def call(self, inputs):
        return self.user_model(inputs["review_username"])

class BeerRetrievalModelV2(tfrs.models.Model):
    def __init__(self,unique_beer_ids, unique_beer_styles, embedding_dimension):
        super().__init__()
        self.beer_model: tf.keras.Model = tf.keras.Sequential([
            tf.keras.layers.IntegerLookup(vocabulary=unique_beer_ids, mask_token=None, name="beer_id_lookup"),
            tf.keras.layers.Embedding(len(unique_beer_ids) + 1, embedding_dimension, name="beer_id_embedding")
        ])
        self.beer_style_model: tf.keras.Model = tf.keras.Sequential([
            tf.keras.layers.StringLookup(vocabulary=unique_beer_styles, mask_token=None,name="beer_style_lookup"),
            tf.keras.layers.Embedding(len(unique_beer_styles) + 1, embedding_dimension, name="beer_style_embedding")
        ])

    def call(self, inputs):
        return tf.concat([
            self.beer_model(inputs["beer_id"]),
            self.beer_style_model(inputs["beer_style"]),
        ], axis=1)

class BeerRecommendationModelV2(tfrs.models.Model):
    def __init__(self,dicts, beers):
        super().__init__()
        self.query_model = tf.keras.Sequential([
          UserRetrievalModelV2(dicts["unique_usernames"],dicts["embedding_dimension"]),
          tf.keras.layers.Dense(dicts["embedding_dimension"])
        ])
        self.candidate_model = tf.keras.Sequential([
          BeerRetrievalModelV2(dicts["unique_beer_ids"], dicts["unique_beer_styles"],dicts["embedding_dimension"]),
          tf.keras.layers.Dense(dicts["embedding_dimension"])
        ])
        self.task = tfrs.tasks.Retrieval(
            metrics=tfrs.metrics.FactorizedTopK(
                candidates=beers.batch(1024).map(self.candidate_model),
            ),
        )

    def compute_loss(self, features, training=False):
        query_embeddings = self.query_model({
            "review_username": features["review_username"]
        })
        candidate_embeddings = self.candidate_model({
            "beer_id": features["beer_id"],
            "beer_style": features["beer_style"]})

        return self.task(query_embeddings, candidate_embeddings)
    def get_model_name(self):
        return "BeerRetrievalV2"

In [19]:
class RankingModelV1(tf.keras.Model):

  def __init__(self, unique_usernames, unique_beer_ids, embedding_dimension):
    super().__init__()

    # Compute embeddings for users.
    self.username_embeddings = tf.keras.Sequential([
      tf.keras.layers.StringLookup(
        vocabulary=unique_usernames, mask_token=None),
      tf.keras.layers.Embedding(len(unique_usernames) + 1, embedding_dimension)
    ])

    # Compute embeddings for beers.
    self.beer_id_embeddings = tf.keras.Sequential([
      tf.keras.layers.IntegerLookup(
        vocabulary=unique_beer_ids, mask_token=None),
      tf.keras.layers.Embedding(len(unique_beer_ids) + 1, embedding_dimension)
    ])

    # Compute predictions.
    self.ratings = tf.keras.Sequential([
      # Learn multiple dense layers.
      tf.keras.layers.Dense(256, activation="relu"),
      tf.keras.layers.Dense(64, activation="relu"),
      # Make rating predictions in the final layer.
      tf.keras.layers.Dense(1)
  ])

  def call(self, inputs):

    username, beer_id = inputs

    username_embedding = self.username_embeddings(username)
    beerid_embedding = self.beer_id_embeddings(beer_id)

    return self.ratings(tf.concat([username_embedding, beerid_embedding], axis=1))

class BeerRankingModelV1(tfrs.models.Model):

  def __init__(self, dicts):
    super().__init__()
    self.ranking_model: tf.keras.Model = RankingModelV1(dicts['unique_usernames'], dicts['unique_beer_ids'], dicts['embedding_dimension'])
    self.task: tf.keras.layers.Layer = tfrs.tasks.Ranking(
      loss = tf.keras.losses.MeanSquaredError(),
      metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

  def call(self, features: Dict[str, tf.Tensor]) -> tf.Tensor:
    return self.ranking_model(
        (features["review_username"], features["beer_id"]))

  def compute_loss(self, features: Dict[Text, tf.Tensor], training=False) -> tf.Tensor:
    labels = features.pop("review_overall")

    rating_predictions = self(features)

    # The task computes the loss and the metrics.
    return self.task(labels=labels, predictions=rating_predictions)
  def get_model_name(self):
        return "BeerRankingModelV1"

In [55]:
# Model ranking
model_ranking = BeerRankingModelV1(embedding_dicts)
model_ranking.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.01))

print('Model name:', model_ranking.get_model_name())

callback = tf.keras.callbacks.EarlyStopping(monitor='root_mean_squared_error', patience=3, min_delta = 0.0005)
ranking_training_results = model_ranking.fit(cached_train, epochs=50, verbose=1, callbacks=callback)
ranking_test_result = model_ranking.evaluate(cached_test, return_dict=True, verbose=1)


Model name: BeerRankingModelV1
Epoch 1/50
9/9 [==============================] - 1s 36ms/step - root_mean_squared_error: 3.3425 - loss: 10.5720 - regularization_loss: 0.0000e+00 - total_loss: 10.5720
Epoch 2/50
9/9 [==============================] - 0s 34ms/step - root_mean_squared_error: 2.0807 - loss: 3.8730 - regularization_loss: 0.0000e+00 - total_loss: 3.8730
Epoch 3/50
9/9 [==============================] - 0s 34ms/step - root_mean_squared_error: 1.0014 - loss: 0.9595 - regularization_loss: 0.0000e+00 - total_loss: 0.9595
Epoch 4/50
9/9 [==============================] - 0s 34ms/step - root_mean_squared_error: 0.8577 - loss: 0.7353 - regularization_loss: 0.0000e+00 - total_loss: 0.7353
Epoch 5/50
9/9 [==============================] - 0s 36ms/step - root_mean_squared_error: 0.8510 - loss: 0.7245 - regularization_loss: 0.0000e+00 - total_loss: 0.7245
Epoch 6/50
9/9 [==============================] - 0s 35ms/step - root_mean_squared_error: 0.8462 - loss: 0.7164 - regularization_los

9/9 [==============================] - 0s 5ms/step - root_mean_squared_error: 0.5542 - loss: 0.3063 - regularization_loss: 0.0000e+00 - total_loss: 0.3063


In [56]:
print(model_ranking({
        "review_username": np.array(["azlondon"]),
        "beer_id": np.array([2468])            
    }))
# Save (deploy) ranking model 
model_ranking.save('model/ranking_model')

tf.Tensor([[3.0873358]], shape=(1, 1), dtype=float32)


INFO:tensorflow:Assets written to: model/ranking_model\assets


INFO:tensorflow:Assets written to: model/ranking_model\assets


In [22]:
# Model retrieval
model_retrieval = BeerRecommendationModelV2(embedding_dicts,beers)
model_retrieval.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.01))

print('Model name:',  model_retrieval.get_model_name())

callback = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=3, min_delta = 100)

retrieval_training_results = model_retrieval.fit(cached_train, epochs=20, verbose=1, callbacks=callback)

retrieval_test_result = model_retrieval.evaluate(cached_test, return_dict=True, verbose=1)


Model name: BeerRetrievalV2
Epoch 1/20
9/9 [==============================] - 46s 4s/step - factorized_top_k/top_1_categorical_accuracy: 0.0035 - factorized_top_k/top_5_categorical_accuracy: 0.0245 - factorized_top_k/top_10_categorical_accuracy: 0.0515 - factorized_top_k/top_50_categorical_accuracy: 0.2609 - factorized_top_k/top_100_categorical_accuracy: 0.5169 - loss: 258755.0430 - regularization_loss: 0.0000e+00 - total_loss: 258755.0430
Epoch 2/20
9/9 [==============================] - 31s 3s/step - factorized_top_k/top_1_categorical_accuracy: 0.0060 - factorized_top_k/top_5_categorical_accuracy: 0.0353 - factorized_top_k/top_10_categorical_accuracy: 0.0709 - factorized_top_k/top_50_categorical_accuracy: 0.3307 - factorized_top_k/top_100_categorical_accuracy: 0.6116 - loss: 257542.5211 - regularization_loss: 0.0000e+00 - total_loss: 257542.5211
Epoch 3/20
9/9 [==============================] - 31s 3s/step - factorized_top_k/top_1_categorical_accuracy: 0.0051 - factorized_top_k/top_5

In [51]:
# Create a model that takes in raw query features, and
index = tfrs.layers.factorized_top_k.BruteForce(model_retrieval.query_model)
# We map the candidate_model over the beer dataset to transform the raw features into embeddings
index.index_from_dataset(
  tf.data.Dataset.zip((beers.batch(100).map(lambda x: x["beer_id"]), beers.batch(100).map(model_retrieval.candidate_model)))
)
#Try suggestion
print(index({"review_username": tf.constant(["azlondon"])}))
# Save (deploy) retrieval model 
tf.saved_model.save(index, 'model/retrieval_model')

(<tf.Tensor: shape=(1, 10), dtype=float32, numpy=
array([[1.4813322, 1.34482  , 1.3428559, 1.3350759, 1.3341382, 1.3331326,
        1.3174257, 1.3032727, 1.293546 , 1.2918501]], dtype=float32)>, <tf.Tensor: shape=(1, 10), dtype=int64, numpy=
array([[2468,  670, 1762, 2225, 5107, 3658,  288,  294, 1316,   51]],
      dtype=int64)>)


INFO:tensorflow:Assets written to: model/retrieval_model\assets


INFO:tensorflow:Assets written to: model/retrieval_model\assets


In [76]:
# Content based filtering
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

# Reset index so that it can be found through matrix
df_beers.reset_index(drop=True, inplace=True)
# Extract relevant columns
df_features = df_beers[['beer_id', 'beer_name', 'beer_style', 'beer_abv']]

# Combine text features
df_features['text_features'] = df_features['beer_name'] + ' ' + df_features['beer_style']

# Vectorize text features using TF-IDF
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
text_features_matrix = tfidf_vectorizer.fit_transform(df_features['text_features'])

# Normalize numerical features
numerical_features = df_features[['beer_abv']].values
scaler = MinMaxScaler()
numerical_features_scaled = scaler.fit_transform(numerical_features)

# Combine text and numerical features
combined_features_matrix = pd.concat([
    pd.DataFrame(text_features_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out()),
    pd.DataFrame(numerical_features_scaled, columns=['beer_abv'])
], axis=1)

# Compute cosine similarity
cosine_sim_matrix = cosine_similarity(combined_features_matrix)

def get_similar_beers(beer_id, cosine_sim_matrix, df_features, top_n=10):
    # Get index of the beer_id
    beer_index = df_features[df_features['beer_id'] == beer_id].index[0]
    
    # Get cosine similarity scores for the given beer
    sim_scores = list(enumerate(cosine_sim_matrix[beer_index]))
    
    # Sort beers based on similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Get top-n similar beers
    sim_beers = sim_scores[1:top_n+1]
    
    # Get beer_ids and beer_names of similar beers
    similar_beers = [(df_features['beer_id'].iloc[i]) for i, _ in sim_beers]
    
    return similar_beers

# Example usage
beer_id = 51 
similar_beers = get_similar_beers(beer_id, cosine_sim_matrix, df_features)
print(f"Beers similar to beer {beer_id}: {similar_beers}")

Beers similar to beer 51: [2205, 3022, 52, 53, 2531, 5923, 30838, 675, 422, 690]


C:\Users\Baris\AppData\Local\Temp\ipykernel_17760\2142554823.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_features['text_features'] = df_features['beer_name'] + ' ' + df_features['beer_style']


In [67]:
#Save cosine matrix 
np.save('model/cosine_similarity_matrix.npy', cosine_sim_matrix)
df_features.to_csv('model/df_features.csv', index=False)



#### Using deployed model
In the next section, we are going to load the model and build the recommedation for given user.

In [70]:
retrieval_model = tf.saved_model.load('model/retrieval_model')
ranking_model = tf.saved_model.load('model/ranking_model')
cosine_sim_matrix = np.load('model/cosine_similarity_matrix.npy')
df_features = pd.read_csv('model/df_features.csv')



In [71]:
#Test deployed retrieval mopdel
retrieval_model({"review_username": ["azlondon"]})

(<tf.Tensor: shape=(1, 10), dtype=float32, numpy=
 array([[1.4813322, 1.34482  , 1.3428559, 1.3350759, 1.3341382, 1.3331326,
         1.3174257, 1.3032727, 1.293546 , 1.2918501]], dtype=float32)>,
 <tf.Tensor: shape=(1, 10), dtype=int64, numpy=
 array([[2468,  670, 1762, 2225, 5107, 3658,  288,  294, 1316,   51]],
       dtype=int64)>)

In [72]:
#Test deployed randing model, the rest of the data should be passed.
input_data = {
    "beer_id": tf.constant([2468], dtype=tf.int64),  # example beer ID
    "review_username": tf.constant(["azlondon"], dtype=tf.string),  # example username
    # Add dummy data for the rest of the inputs
    "beer_abv": tf.constant([0.0], dtype=tf.float32),  # assuming float type
    "beer_name": tf.constant(["dummy_beer_name"], dtype=tf.string),
    "beer_style": tf.constant(["dummy_beer_style"], dtype=tf.string),
    "brewer_id": tf.constant([0], dtype=tf.int64),  # assuming integer type
    "review_appearance": tf.constant([0.0], dtype=tf.float32),
    "review_aroma": tf.constant([0.0], dtype=tf.float32),
    "review_palate": tf.constant([0.0], dtype=tf.float32),
    "review_taste": tf.constant([0.0], dtype=tf.float32),
}
ranking_model(input_data)

<tf.Tensor: shape=(1, 1), dtype=float32, numpy=array([[3.0873358]], dtype=float32)>

In [74]:
#Test deployed content based filtering
print(get_similar_beers(2468, cosine_sim_matrix, df_features, top_n=20));

[(647, 'Paulaner Hefe-Weissbier'), (1156, 'Weihenstephaner Hefe Weissbier'), (1088, 'Franziskaner Hefe-Weissbier'), (1762, 'Schneider Weisse Original'), (2224, 'Schneider Aventinus'), (690, 'Boddingtons Pub Ale &#40;Can&#41;'), (365, 'Sierra Nevada Pale Ale &#40;Bottle&#41;'), (422, 'Stone India Pale Ale &#40;IPA&#41;'), (10514, 'Schneider Aventinus Weizen-Eisbock'), (51, 'Chimay Rouge &#40;Red&#41; / Premire'), (53, 'Chimay Bleue &#40;Blue&#41; / Grande Rserve'), (30838, 'Stone Old Guardian &#40;Vintages 2004 and later&#41;'), (405, 'Miller Genuine Draft &#40;MGD&#41;'), (5107, 'Wychwood Hobgoblin &#40;Pasteurised&#41;'), (52, 'Chimay Triple / Blanche &#40;White&#41; / Cinq Cents'), (4456, 'Guinness Extra Stout &#40;North America&#41;'), (5923, 'Dogfish Head World Wide Stout 2001/2003-Present &#40;18%&#41;'), (1316, 'Budweiser Budvar &#40;Czechvar&#41; 12'), (303, 'Fullers London Porter &#40;Bottle/Keg&#41;'), (52930, 'Anchor Our Special Ale &#40;2005 and later&#41;')]


In [110]:
def get_recommendation_for(username, top_n=10):
    #define recommended beer ids
    recommended_beer_ids = []
    #get beer ids for username from retrieval model
    retrieval_results = retrieval_model({"review_username": [username]})
    recommended_retrieval_beer_ids = retrieval_results[1].numpy()[0]
    for retrieval_beer_id in recommended_retrieval_beer_ids:
        recommended_beer_ids.append(retrieval_beer_id)
        #get similar beers for the recommended beer id from retrieval model
        similar_beer_ids = get_similar_beers(retrieval_beer_id, cosine_sim_matrix, df_features, top_n=10)
        recommended_beer_ids.extend(similar_beer_ids)
    #remove duplicated ids
    recommended_beer_ids = set(recommended_beer_ids)
    undiscovered_beers = []
    #filter out beer ids that already in username's reviewed beers
    for recommended_beer_id in recommended_beer_ids:
        has_review_already = ((df['review_username'] == username) & (df['beer_id'] == recommended_beer_id)).any()
        if(not has_review_already):
            #Get estimated rating from the ranking model
            input_data = {
                "beer_id": tf.constant([recommended_beer_id], dtype=tf.int64),  # example beer ID
                "review_username": tf.constant([username], dtype=tf.string),  # example username
                # Add dummy data for the rest of the inputs
                "beer_abv": tf.constant([0.0], dtype=tf.float32),  # assuming float type
                "beer_name": tf.constant(["dummy_beer_name"], dtype=tf.string),
                "beer_style": tf.constant(["dummy_beer_style"], dtype=tf.string),
                "brewer_id": tf.constant([0], dtype=tf.int64),  # assuming integer type
                "review_appearance": tf.constant([0.0], dtype=tf.float32),
                "review_aroma": tf.constant([0.0], dtype=tf.float32),
                "review_palate": tf.constant([0.0], dtype=tf.float32),
                "review_taste": tf.constant([0.0], dtype=tf.float32),
            }
            estimated_rating = ranking_model(input_data)[0][0].numpy()
            recommended_beer = df_beers[df_beers['beer_id'] == recommended_beer_id].iloc[0]
            undiscovered_beers.append({
                "beer_id": recommended_beer_id,
                "brewer_id": recommended_beer["brewer_id"],
                "beer_name": recommended_beer['beer_name'],
                "beer_style": recommended_beer['beer_style'],
                "beer_abv": recommended_beer["beer_abv"],
                "estimated_rating": estimated_rating
            })
    #sort by estimated ratings and return top_n beers
    return sorted(undiscovered_beers, key=lambda x: x['estimated_rating'], reverse=True)[:top_n]
    
            


recommended_beers = get_recommendation_for("AndySnow")
recommended_beers

[{'beer_id': 15917,
  'brewer_id': 231,
  'beer_name': 'Three Floyds Dark Lord Russian Imperial Stout',
  'beer_style': 'Imperial Stout',
  'beer_abv': 15.0,
  'estimated_rating': 4.368179},
 {'beer_id': 4935,
  'brewer_id': 623,
  'beer_name': 'Westvleteren Extra 8',
  'beer_style': 'Belgian Strong Ale',
  'beer_abv': 8.0,
  'estimated_rating': 4.2856483},
 {'beer_id': 5923,
  'brewer_id': 198,
  'beer_name': 'Dogfish Head World Wide Stout 2001/2003-Present &#40;18%&#41;',
  'beer_style': 'Imperial Stout',
  'beer_abv': 18.0,
  'estimated_rating': 4.192251},
 {'beer_id': 422,
  'brewer_id': 76,
  'beer_name': 'Stone India Pale Ale &#40;IPA&#41;',
  'beer_style': 'India Pale Ale (IPA)',
  'beer_abv': 6.9,
  'estimated_rating': 4.0788283},
 {'beer_id': 3213,
  'brewer_id': 232,
  'beer_name': 'Third Coast Old Ale',
  'beer_style': 'Barley Wine',
  'beer_abv': 10.2,
  'estimated_rating': 4.035212},
 {'beer_id': 6862,
  'brewer_id': 1163,
  'beer_name': 'De Dolle Stille Nacht',
  'beer_st

In [112]:
df[df["review_username"] == "Vaiz"].sort_values(by="review_overall", ascending=False)

,beer_name,beer_id,brewer_id,beer_abv,beer_style,review_appearance,review_aroma,review_palate,review_taste,review_overall,review_time,review_username,review_text,review_count
68242,Orval,835,132,6.2,Belgian Ale,4.0,4.5,5.0,4.5,4.75,1260835200,Vaiz,"UPDATED: JAN 1, 2011 Pours a hazy orange with ...",2353
36014,Hair of the Dog Adam,568,94,10.0,Traditional Ale,4.0,4.0,3.0,4.5,4.75,1306281600,Vaiz,Bottle at home. Dark brown to reddish color wi...,1313
278172,Westvleteren 12,4934,623,10.2,Abt/Quadrupel,4.0,3.5,5.0,4.5,4.50,1268092800,Vaiz,"Bottle at home, from a box retrieved at the ab...",1608
239711,Three Floyds Dark Lord Russian Imperial Stout,15917,231,15.0,Imperial Stout,3.0,4.0,4.0,4.0,4.50,1280534400,Vaiz,Bottle at Dutch RB Meeting. Thanks a lot for s...,1191
247098,Russian River Pliny the Elder,8936,1480,8.0,Imperial/Double IPA,3.0,4.5,4.0,4.5,4.50,1293753600,Vaiz,"Bottle at home, thanks Santa! Cloudy orange po...",1370
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
308102,Fosters Lager,187,38,4.9,Pale Lager,2.0,2.0,2.0,1.5,1.75,1297468800,Vaiz,"Clear bottle at home. Golden pour, thin white ...",1212
249132,Heineken,37,9,5.0,Pale Lager,2.0,2.0,2.0,1.5,1.25,1261699200,Vaiz,"Pale golden color, small white head. Slightly ...",1929
140580,Miller Genuine Draft &#40;MGD&#41;,405,75,4.7,Pale Lager,2.0,1.0,2.0,1.0,1.00,1317427200,Vaiz,"Bottle at home. Clear golden pour, no head. Gr...",1323
188593,Budweiser,473,84,5.0,Pale Lager,2.0,1.0,1.0,1.5,1.00,1258848000,Vaiz,Almost colorless watery looking beer with a th...,1929


In [111]:
df

,beer_name,beer_id,brewer_id,beer_abv,beer_style,review_appearance,review_aroma,review_palate,review_taste,review_overall,review_time,review_username,review_text,review_count
0,Chimay Rouge &#40;Red&#41; / Premire,51,12,7.0,Abbey Dubbel,3.0,3.5,3.0,4.0,3.50,1214524800,azlondon,Appearance: Dark redish brown. Little head.\tA...,2062
1,Chimay Rouge &#40;Red&#41; / Premire,51,12,7.0,Abbey Dubbel,3.0,3.5,4.0,3.5,3.50,1214179200,blipp,Bottle. Pours cloudy brown with a fizzy beige ...,2062
2,Chimay Rouge &#40;Red&#41; / Premire,51,12,7.0,Abbey Dubbel,4.0,3.5,2.0,4.0,2.75,1214092800,uhre,"Pours brown colour with a tint of red, the hea...",2062
3,Chimay Rouge &#40;Red&#41; / Premire,51,12,7.0,Abbey Dubbel,3.0,2.5,4.0,4.0,4.25,1213747200,metsbcd,"Very good beer, had it at a Mortons. Appearanc...",2062
4,Chimay Rouge &#40;Red&#41; / Premire,51,12,7.0,Abbey Dubbel,3.0,4.0,3.0,3.5,4.25,1213660800,Joakgust,33 cl bottle - Beer looks very much like the b...,2062
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
308098,Fosters Lager,187,38,4.9,Pale Lager,3.0,2.0,2.0,2.0,1.50,1299283200,arminjewell,"Pours clear yellow, fizzy white head. Aroma o...",1212
308099,Fosters Lager,187,38,4.9,Pale Lager,1.0,1.0,1.0,1.0,0.75,1298419200,txspartan,I am looking forward to drinking a decent Aust...,1212
308100,Fosters Lager,187,38,4.9,Pale Lager,2.0,1.5,3.0,1.5,1.75,1298246400,AndySnow,Light yellow colour. Neutral taste. Average la...,1212
308101,Fosters Lager,187,38,4.9,Pale Lager,4.0,4.0,4.0,3.5,3.50,1298160000,Tricountybeer,poured from a 12 ounce bottle brewed from oil...,1212
